- *Can we identify parallel roles purely from network structure, without narrative context?* SØREN
    * Notes:
        - Node2vec algorithm for finding nodes which are similar in one universe.
        - Struc2vec algorithm for finding nodes across networks which occupy similar structural positions.
        - Maybe: Compute a vector of structural properties (In-degree, Out-degree, Closeness, Eigenvector / pageRank, Clustering coefficient, triadic position, etc.) for each characters, and use clustering algorithms (k-means, hierarchical clustering, Gaussian mixture models)

In [ ]:
# Create the edgelist for the stuc2vec algorithm

import numpy as np
import networkx as nx 
import pickle
from util import *

with open("hp_characters.pkl", "rb") as f:   # 'rb' = read binary
    HP_data = pickle.load(f)
    
HP_network = create_network(HP_data, field_origin = 'house', field_species = 'species')

with open("lotr_characters.pkl", "rb") as f:   # 'rb' = read binary
    LOTR_data = pickle.load(f)

LOTR_network = create_network(LOTR_data, field_origin = 'culture', field_species = 'race')

G = nx.union(HP_network, LOTR_network)
mapping = {old_label: i + 1 for i, old_label in enumerate(G.nodes())}
G = nx.relabel_nodes(G, mapping)
#nx.write_edgelist(G, "C:\\Users\\soere\\OneDrive\\Skrivebord\\struc2vec-master\\struc2vec-master\\graph\\graph.edgelist", data=False)

#with open("mapping.pkl", "wb") as f:
#    pickle.dump(mapping, f)


In [ ]:
with open("mapping.pkl", "rb") as f:
    mapping = pickle.load(f)
# Load the generated embedding
from gensim.models import KeyedVectors
embeddings = KeyedVectors.load_word2vec_format(f"embedding.emb", binary=False)

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

# -----------------------------------------------------------
# Step 1: Function to select best number of GMM clusters via BIC
# -----------------------------------------------------------

def select_best_gmm_k(X, k_min=2, k_max=15):
    bic_scores = []
    gmms = []
    
    for k in range(k_min, k_max + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            random_state=42
        )
        gmm.fit(X)
        bic = gmm.bic(X)
        bic_scores.append(bic)
        gmms.append(gmm)

    best_idx = np.argmin(bic_scores)
    best_k = k_min + best_idx
    best_gmm = gmms[best_idx]
    
    print(f"Best number of clusters (BIC): {best_k}")
    return best_k, best_gmm, bic_scores

In [ ]:
def construct_X(G, embedding):
    in_deg = np.array([deg for _, deg in G.in_degree()])
    out_deg = np.array([deg for _, deg in G.out_degree()])
    closesness = np.array(list(nx.closeness_centrality(G).values()))
    pagerank = np.array(list(nx.pagerank(G).values()))
    clustering = np.array(list(nx.clustering(G).values()))

    word_vectors = []

    for key in embedding.index_to_key:
        word_vectors.append((int(key), embedding[key]))

    word_vectors = sorted(word_vectors, key = lambda x: x[0])

    word_vectors = np.array([snd for _, snd in word_vectors])

    X = np.c_[np.array([in_deg, out_deg, closesness, pagerank, clustering]).T, word_vectors]
    return X

import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def plot_pca_first_two(Z, labels):
    """
    Plots the first two PCA directions of Z, colored by labels.
    
    Parameters
    ----------
    Z : np.ndarray
        Normalized data matrix of shape (n_samples, n_features)
    labels : array-like
        Vector of group labels (length n_samples)
    """
    # PCA → 2D projection
    pca = PCA(n_components=2)
    Z_pca = pca.fit_transform(Z)

    # Convert labels to a numpy array
    labels = np.array(labels)

    # Unique groups
    unique_labels = np.unique(labels)

    plt.figure(figsize=(7, 7))

    # Plot each group separately to color them differently
    for lab in unique_labels:
        idx = labels == lab
        plt.scatter(
            Z_pca[idx, 0],
            Z_pca[idx, 1],
            label=str(lab)
        )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("PCA (first two components)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
X = construct_X(G, embeddings)
Z = (X - np.mean(X, axis = 0)) / np.std(X, axis = 0)

In [ ]:
best_k, best_gmm, bic_scores = select_best_gmm_k(Z)

result = best_gmm.fit_predict(Z)
reversed_mapping = {value : item for item, value in mapping.items()}
groups = {}
for x in set(result):
    idx = np.where(result == x)[0] + 1
    lst = [reversed_mapping[i] for i in idx]
    groups[x] = {'Names' : lst, 'index' : idx}

plot_pca_first_two(Z, result)

In [ ]:
groups

In [ ]:
Z_main = Z[groups[0]['index'],:]
_, best_gmm_main, _ = select_best_gmm_k(Z_main)
result_main = best_gmm_main.predict(Z_main)
reversed_mapping = {value : item for item, value in mapping.items()}
groups_main = {}
for x in set(result_main):
    idx = np.where(result_main == x)[0] + 1
    lst = [reversed_mapping[i] for i in idx]
    groups_main[x] = {'Names' : lst, 'index' : idx}
plot_pca_first_two(Z_main, result_main)



In [ ]:
groups_main